# CellPress & Nature Crawling Tool — Step-by-Step Tutorial

This notebook walks you through every feature of the **`papers_crawler`** package, which can:

| Target | What you get |
|--------|-------------|
| **Cell.com (CellPress)** | Download **PDFs** *and/or* extract **full-text JSON** |
| **Nature.com** | Extract **full-text JSON** (structured article content) |

---

## Table of Contents
1. [Installation & Setup](#1-installation--setup)
2. [Package overview & available functions](#2-package-overview--available-functions)
3. [CellPress — Discover available journals](#3-cellpress--discover-available-journals)
4. [CellPress — Download PDFs (sync)](#4-cellpress--download-pdfs-sync)
5. [CellPress — Download PDFs (async / Jupyter-friendly)](#5-cellpress--download-pdfs-async--jupyter-friendly)
6. [CellPress — Extract full-text JSON (async)](#6-cellpress--extract-full-text-json-async)
7. [Nature — Discover available journals](#7-nature--discover-available-journals)
8. [Nature — Extract full-text JSON (async)](#8-nature--extract-full-text-json-async)
9. [Parameter reference cheatsheet](#9-parameter-reference-cheatsheet)
10. [Working with the output files](#10-working-with-the-output-files)
11. [Progress callbacks — custom hooks](#11-progress-callbacks--custom-hooks)
12. [Troubleshooting](#12-troubleshooting)


---
## 1 — Installation & Setup

### 1.1 Install Python dependencies

Run the cell below **once** to install the package and all its requirements.  
Inside a virtual-env or Colab, add `--quiet` to suppress verbose output.

In [ ]:
# Install from the local source tree (recommended when working inside the repo)
%pip install -e . --quiet

# -- OR -- install directly from PyPI / GitHub once published:
# %pip install papers-crawler

### 1.2 Install the Playwright browser

The crawlers use **Firefox** via Playwright.  
Run this **once per environment** (Colab, Docker, local machine …).

In [ ]:
import subprocess, sys

# Install Firefox browser binaries used by Playwright
result = subprocess.run(
    [sys.executable, "-m", "playwright", "install", "firefox"],
    capture_output=True, text=True
)
print(result.stdout or "Browser already installed.")
if result.returncode != 0:
    print("STDERR:", result.stderr)

### 1.3 Verify the installation

In [ ]:
import papers_crawler
print("papers_crawler version:", papers_crawler.__version__)
print("Available exports:", dir(papers_crawler))

---
## 2 — Package overview & available functions

The package exposes the following top-level functions after `import papers_crawler`:

| Function | Source module | Purpose |
|----------|---------------|---------|
| `crawl` | `crawler` | **Sync** PDF download from Cell.com |
| `crawl_async` | `crawler_async` | **Async** PDF download (Jupyter / Colab) |
| `crawl_text_async` | `crawl_text_async` | **Async** full-text JSON from Cell.com |
| `crawl_text_nature_async` | `crawl_text_async_nature` | **Async** full-text JSON from Nature.com |
| `discover_journals` | `crawler` | List all Cell.com journals (sync) |
| `discover_journals_async` | `crawler_async` | List all Cell.com journals (async) |
| `discover_journals_nature_async` | `crawl_text_async_nature` | List all Nature.com journals (async) |


In [ ]:
# Import everything you'll need for this tutorial
from papers_crawler import (
    crawl,                          # sync CellPress PDF downloader
    crawl_async,                    # async CellPress PDF downloader
    crawl_text_async,               # async CellPress full-text extractor
    crawl_text_nature_async,        # async Nature full-text extractor
    discover_journals,              # sync journal discovery (CellPress)
    discover_journals_async,        # async journal discovery (CellPress)
    discover_journals_nature_async, # async journal discovery (Nature)
)

import asyncio, json, os, pathlib, pandas as pd
print("All imports successful.")

---
## 3 — CellPress — Discover available journals

Before crawling you often want to know which **journal slugs** exist.  
A *slug* is the short identifier used in the URL (e.g. `immunity`, `cell`, `joule`).

### 3.1 Sync discovery

In [ ]:
# discover_journals() → List[Tuple[str, str]]
#   Returns a list of (slug, display_name) tuples.
#
#   Parameters:
#     force_refresh (bool, default False)
#       False → use cached list (fast, no network call)
#       True  → re-fetch from Cell.com and refresh the cache

cellpress_journals = discover_journals(force_refresh=False)

print(f"Found {len(cellpress_journals)} CellPress journals.\n")
print(f"{'Slug':<30} {'Display name'}")
print("-" * 60)
for slug, name in cellpress_journals[:20]:   # show first 20
    print(f"{slug:<30} {name}")
print("...")

In [ ]:
# Put them in a DataFrame for easier filtering
df_cp = pd.DataFrame(cellpress_journals, columns=["slug", "name"])
df_cp.head(10)

In [ ]:
# Search for journals whose name contains a keyword
query = "cell"
matches = df_cp[df_cp["name"].str.contains(query, case=False)]
print(f"Journals matching '{query}':")
print(matches.to_string(index=False))

### 3.2 Async discovery (preferred inside Jupyter)

In [ ]:
# Jupyter already runs an event loop, so use await directly.
# In a plain Python script use: asyncio.run(discover_journals_async())

# discover_journals_async(force_refresh=False) → same result as the sync version
cellpress_journals_async = await discover_journals_async(force_refresh=False)
print(f"Async discovery returned {len(cellpress_journals_async)} journals.")

---
## 4 — CellPress — Download PDFs (sync)

> **Use this in a normal Python script.** Inside Jupyter, prefer the async version (Section 5).

### Function signature

```python
crawl(
    keywords      : str            = "",       # search query – leave empty to get all articles
    year_from     : int            = 2020,     # earliest publish year (inclusive)
    year_to       : int            = 2024,     # latest publish year  (inclusive)
    out_folder    : str            = "papers", # directory where PDFs are saved
    headless      : bool           = True,     # True  → invisible browser (faster)
                                               # False → visible browser (useful for debugging)
    limit         : Optional[int]  = None,     # max articles per journal; None = unlimited
    journal_slugs : Optional[List[str]] = None,# list of slugs to crawl; None = ALL journals
    progress_callback       = None,            # called with (filename, filepath)
    total_progress_callback = None,            # called with (current, total, status, …)
) -> Tuple[List[str], List[str]]              # (pdf_file_paths, article_titles)
```

### 4.1 Minimal example — one journal, 3 articles

In [ ]:
# NOTE: This cell uses the SYNC crawler.
# In Jupyter it will block the kernel until done — that's expected.

file_paths, titles = crawl(
    keywords      = "",           # no keyword filter → retrieve all matching dates
    year_from     = 2023,
    year_to       = 2024,
    out_folder    = "papers",     # PDFs will land in ./papers/immunity/
    headless      = True,
    limit         = 3,            # stop after 3 articles (great for testing)
    journal_slugs = ["immunity"], # only crawl the Immunity journal
)

print(f"Downloaded {len(file_paths)} file(s).")
for fp, title in zip(file_paths, titles):
    size_kb = os.path.getsize(fp) / 1024 if os.path.exists(fp) else 0
    print(f"  [{size_kb:6.1f} KB] {title}")

### 4.2 Keyword filtering example

In [ ]:
# Use keywords to restrict results to articles that match the search terms
file_paths, titles = crawl(
    keywords      = "T cell immune response",  # search phrase
    year_from     = 2022,
    year_to       = 2024,
    out_folder    = "papers",
    headless      = True,
    limit         = 5,
    journal_slugs = ["immunity", "cell"],       # search across two journals
)

print(f"Downloaded {len(file_paths)} article(s) matching the keyword.")
for title in titles:
    print(" •", title)

### 4.3 Crawl all journals (no slug filter)

> **Warning:** Omitting `journal_slugs` will crawl *every* CellPress journal — potentially thousands of articles. Always set a sensible `limit` when testing.

In [ ]:
# Crawl ALL CellPress journals, at most 2 articles each, for year 2024
file_paths, titles = crawl(
    keywords      = "",
    year_from     = 2024,
    year_to       = 2024,
    out_folder    = "papers_all",
    headless      = True,
    limit         = 2,            # 2 per journal → manageable
    journal_slugs = None,         # None = ALL journals
)

print(f"Total downloaded: {len(file_paths)}")

---
## 5 — CellPress — Download PDFs (async / Jupyter-friendly)

`crawl_async` is the **recommended** way to download PDFs inside a Jupyter notebook or Google Colab because it works with the running event loop.

### Function signature

```python
await crawl_async(
    keywords       : str            = "",
    year_from      : int            = 2020,
    year_to        : int            = 2024,
    out_folder     : str            = "papers",
    headless       : bool           = True,
    limit          : Optional[int]  = None,
    journal_slugs  : Optional[List[str]] = None,
    progress_callback       = None,
    total_progress_callback = None,
    crawl_archives : bool           = False,  # also scrape /issue archive pages
) -> Tuple[List[str], List[str]]
```

### 5.1 Basic async download

In [ ]:
# Inside Jupyter use 'await' directly — no asyncio.run() needed.

file_paths, titles = await crawl_async(
    keywords      = "",
    year_from     = 2023,
    year_to       = 2024,
    out_folder    = "papers",
    headless      = True,
    limit         = 3,
    journal_slugs = ["immunity"],
)

print("Files saved:")
for fp in file_paths:
    print(" •", fp)

### 5.2 Enable archive crawling (`crawl_archives=True`)

By default only the *latest* issue pages are crawled.  
Setting `crawl_archives=True` also scrapes all `/issue` pages, which may include older articles not surfaced on the main journal page.

In [ ]:
file_paths, titles = await crawl_async(
    keywords       = "",
    year_from      = 2020,
    year_to        = 2021,
    out_folder     = "papers_archive",
    headless       = True,
    limit          = 5,
    journal_slugs  = ["cell"],
    crawl_archives = True,   # <-- include issue archive pages
)

print(f"Downloaded {len(file_paths)} article(s) from archive.")

### 5.3 Async download in a regular Python script

Outside of Jupyter, wrap the call with `asyncio.run()`:

```python
import asyncio
from papers_crawler import crawl_async

async def main():
    file_paths, titles = await crawl_async(
        keywords      = "CRISPR",
        year_from     = 2023,
        year_to       = 2024,
        out_folder    = "papers",
        limit         = 10,
        journal_slugs = ["cell", "immunity"],
    )
    print(f"Done — {len(file_paths)} files.")

if __name__ == "__main__":
    asyncio.run(main())
```

---
## 6 — CellPress — Extract full-text JSON (async)

Instead of (or in addition to) PDFs, you can extract the **full article text** as structured JSON.

### Function signature

```python
await crawl_text_async(
    keywords       : str            = "",
    year_from      : int            = 2020,
    year_to        : int            = 2024,
    out_folder     : str            = "papers",      # JSON files saved here
    headless       : bool           = True,
    limit          : Optional[int]  = None,
    journal_slugs  : Optional[List[str]] = None,
    progress_callback       = None,
    total_progress_callback = None,
    crawl_archives : bool           = False,
) -> Tuple[List[str], List[str]]                    # (json_paths, titles)
```

### 6.1 Basic JSON extraction — Immunity journal

In [ ]:
json_paths, titles = await crawl_text_async(
    keywords      = "",
    year_from     = 2024,
    year_to       = 2024,
    out_folder    = "papers_text",    # separate folder from PDFs
    headless      = True,
    limit         = 2,
    journal_slugs = ["immunity"],
)

print(f"Extracted {len(json_paths)} JSON file(s).")
for jp in json_paths:
    print(" •", jp)

### 6.2 Inspect the JSON structure

In [ ]:
# Load and pretty-print the first extracted article
if json_paths:
    with open(json_paths[0], encoding="utf-8") as f:
        article = json.load(f)

    # Top-level keys
    print("Top-level keys:", list(article.keys()))
    print()

    # Metadata fields
    for key in ["title", "authors", "publication_date", "doi", "url", "extracted_at"]:
        print(f"  {key}: {str(article.get(key, 'N/A'))[:120]}")

    print()
    print("Abstract (first 300 chars):")
    print(" ", str(article.get("Abstract", ""))[:300])

In [ ]:
# List all section headings extracted from the article
if json_paths:
    excluded_meta = {"url", "extracted_at", "title", "authors",
                     "publication_date", "doi", "Abstract", "References", "Figures"}
    sections = [k for k in article.keys() if k not in excluded_meta]
    print("Article sections:")
    for s in sections:
        print(" •", s)

In [ ]:
# Print the beginning of the Introduction section (if it exists)
if json_paths and "Introduction" in article:
    print("Introduction (first 500 chars):")
    print(article["Introduction"][:500])

In [ ]:
# Show reference list
if json_paths and "References" in article:
    refs = article["References"]
    print(f"Total references: {len(refs)}")
    print("First 5 references:")
    for r in refs[:5]:
        print(" ", r)

### 6.3 Build a DataFrame from multiple extracted articles

In [ ]:
records = []
json_dir = pathlib.Path("papers_text")

for fp in json_dir.rglob("*.json"):
    # skip the ZIP summary files
    if "summary" in fp.name or fp.suffix != ".json":
        continue
    try:
        with open(fp, encoding="utf-8") as f:
            art = json.load(f)
        records.append({
            "file"            : fp.name,
            "title"           : art.get("title", ""),
            "authors"         : art.get("authors", ""),
            "publication_date": art.get("publication_date", ""),
            "doi"             : art.get("doi", ""),
            "n_sections"      : len([k for k in art if k not in
                                     {"url","extracted_at","title","authors",
                                      "publication_date","doi"}]),
        })
    except Exception as e:
        print(f"Could not parse {fp}: {e}")

df_articles = pd.DataFrame(records)
print(f"Loaded {len(df_articles)} article(s).")
df_articles.head()

---
## 7 — Nature — Discover available journals

Nature.com hosts hundreds of journals.  
`discover_journals_nature_async` parses the `nature.com/siteindex` page and caches the result.

### Function signature

```python
await discover_journals_nature_async(
    force_refresh : bool = False   # True → re-fetch and update cache
) -> List[Tuple[str, str]]        # (slug, display_name)
```

In [ ]:
nature_journals = await discover_journals_nature_async(force_refresh=False)

print(f"Found {len(nature_journals)} Nature.com journals.\n")
print(f"{'Slug':<35} {'Display name'}")
print("-" * 70)
for slug, name in nature_journals[:20]:
    print(f"{slug:<35} {name}")
print("...")

In [ ]:
# Filter by keyword to find relevant slugs
df_nat = pd.DataFrame(nature_journals, columns=["slug", "name"])

keyword = "cancer"
print(f"Nature journals matching '{keyword}':")
print(df_nat[df_nat["name"].str.contains(keyword, case=False)].to_string(index=False))

---
## 8 — Nature — Extract full-text JSON (async)

> **Note:** Nature.com does not provide direct PDF download links, so this crawler extracts the **structured HTML full-text** and saves it as JSON — identical in format to the CellPress JSON output.

### Function signature

```python
await crawl_text_nature_async(
    keywords       : str            = "",            # reserved for future use
    year_from      : int            = 2020,
    year_to        : int            = 2024,
    out_folder     : str            = "papers_nature",
    headless       : bool           = True,
    limit          : Optional[int]  = None,
    journal_slugs  : Optional[List[str]] = None,     # None = ALL Nature journals
    progress_callback       = None,
    total_progress_callback = None,
    crawl_archives : bool           = False,         # reserved for future use
) -> Tuple[List[str], List[str]]                    # (json_paths, titles)
```

### 8.1 Single journal — Nature Medicine

In [ ]:
nat_json_paths, nat_titles = await crawl_text_nature_async(
    year_from     = 2024,
    year_to       = 2024,
    out_folder    = "papers_nature",
    headless      = True,
    limit         = 2,                     # 2 articles for a quick test
    journal_slugs = ["nature-medicine"],   # slug for Nature Medicine
)

print(f"Extracted {len(nat_json_paths)} article(s).")
for jp, t in zip(nat_json_paths, nat_titles):
    print(f"  [{pathlib.Path(jp).stat().st_size // 1024} KB] {t}")

### 8.2 Multiple Nature journals

In [ ]:
nat_json_paths, nat_titles = await crawl_text_nature_async(
    year_from     = 2023,
    year_to       = 2024,
    out_folder    = "papers_nature",
    headless      = True,
    limit         = 3,
    journal_slugs = [
        "nature",           # flagship Nature journal
        "nature-medicine",
        "nature-cancer",
    ],
)

print(f"Total extracted: {len(nat_json_paths)} article(s).")

### 8.3 Inspect a Nature JSON file

In [ ]:
if nat_json_paths:
    with open(nat_json_paths[0], encoding="utf-8") as f:
        nat_article = json.load(f)

    print("Top-level keys:", list(nat_article.keys()))
    print()
    for key in ["title", "authors", "publication_date", "doi"]:
        val = nat_article.get(key, "N/A")
        print(f"  {key}: {str(val)[:120]}")
    print()
    print("Abstract (first 300 chars):")
    print(" ", str(nat_article.get("Abstract", ""))[:300])

### 8.4 Crawl all Nature journals

Omitting `journal_slugs` (or passing `None`) crawls **all** discovered journals.  
Again, always set `limit` when experimenting.

In [ ]:
# Crawl ALL Nature journals — 1 article each — for 2024
nat_json_paths, nat_titles = await crawl_text_nature_async(
    year_from     = 2024,
    year_to       = 2024,
    out_folder    = "papers_nature_all",
    headless      = True,
    limit         = 1,           # safety limit
    journal_slugs = None,        # ALL Nature journals
)

print(f"Total articles collected: {len(nat_json_paths)}")

---
## 9 — Parameter reference cheatsheet

### Common parameters (all four main functions share these)

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `keywords` | `str` | `""` | Free-text search query. Empty string = no filter. |
| `year_from` | `int` | `2020` | First year to include (inclusive). |
| `year_to` | `int` | `2024` | Last year to include (inclusive). |
| `out_folder` | `str` | `"papers"` | Root output directory. Sub-folders per journal are created automatically. |
| `headless` | `bool` | `True` | `True` → no visible browser window. `False` → show browser (debug mode). |
| `limit` | `int \| None` | `None` | Max articles **per journal**. `None` = unlimited. |
| `journal_slugs` | `list \| None` | `None` | List of journal slugs to crawl. `None` = crawl **all** discovered journals. |
| `progress_callback` | `callable \| None` | `None` | Called with `(filename, filepath)` after each article. |
| `total_progress_callback` | `callable \| None` | `None` | Called with `(current, total, status, file_size_kb, speed_kbps, stage)`. |

### Async-only parameter

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `crawl_archives` | `bool` | `False` | Also scrape `/issue` archive pages for older articles (CellPress only). |

### Return value

All four functions return `(List[str], List[str])` → `(file_paths, article_titles)`

- **`file_paths`** — absolute paths to the saved files (`.pdf` for `crawl` / `crawl_async`; `.json` for text extractors)
- **`article_titles`** — display titles of the downloaded articles

### Common journal slugs

**CellPress**

| Slug | Journal name |
|------|--------------|
| `cell` | Cell |
| `immunity` | Immunity |
| `neuron` | Neuron |
| `cancer-cell` | Cancer Cell |
| `cell-metabolism` | Cell Metabolism |
| `cell-reports` | Cell Reports |
| `current-biology` | Current Biology |
| `developmental-cell` | Developmental Cell |
| `joule` | Joule |
| `matter` | Matter |

**Nature**

| Slug | Journal name |
|------|--------------|
| `nature` | Nature |
| `nature-medicine` | Nature Medicine |
| `nature-cancer` | Nature Cancer |
| `nature-biotechnology` | Nature Biotechnology |
| `nature-genetics` | Nature Genetics |
| `nature-immunology` | Nature Immunology |
| `nature-neuroscience` | Nature Neuroscience |
| `nature-cell-biology` | Nature Cell Biology |
| `nature-methods` | Nature Methods |
| `nature-communications` | Nature Communications |


---
## 10 — Working with the output files

### 10.1 Inspect the output directory tree

In [ ]:
import pathlib

def tree(root, max_depth=3, indent=""):
    """Recursively print directory tree (up to max_depth)."""
    p = pathlib.Path(root)
    if not p.exists():
        print(f"{root} — does not exist yet.")
        return
    for item in sorted(p.iterdir()):
        print(indent + item.name + ("/" if item.is_dir() else ""))
        if item.is_dir() and max_depth > 1:
            tree(item, max_depth - 1, indent + "  ")

print("=== papers/ ===")
tree("papers")
print()
print("=== papers_nature/ ===")
tree("papers_nature")

### 10.2 Read the CSV summary generated by the crawler

Each run produces an `extraction_summary_*.csv` (or `download_summary_*.csv`) inside the output folder.

In [ ]:
import glob as _glob

csv_files = sorted(_glob.glob("papers/**/*.csv", recursive=True) +
                   _glob.glob("papers_nature/**/*.csv", recursive=True))

if csv_files:
    latest_csv = csv_files[-1]
    print("Loading:", latest_csv)
    df_summary = pd.read_csv(latest_csv)
    print(df_summary.head(10))
else:
    print("No summary CSV found yet — run a crawl first.")

### 10.3 Batch-load all JSON files into a DataFrame

In [ ]:
def load_json_articles(root_folder: str) -> pd.DataFrame:
    """Load every *.json article file under root_folder into a DataFrame."""
    records = []
    for fp in pathlib.Path(root_folder).rglob("*.json"):
        if "summary" in fp.stem:
            continue
        try:
            with open(fp, encoding="utf-8") as f:
                art = json.load(f)
            records.append({
                "source"          : root_folder,
                "journal"         : fp.parent.name,
                "file"            : fp.name,
                "title"           : art.get("title", ""),
                "authors"         : art.get("authors", ""),
                "publication_date": art.get("publication_date", ""),
                "doi"             : art.get("doi", ""),
                "abstract"        : art.get("Abstract", ""),
                "url"             : art.get("url", ""),
            })
        except Exception as e:
            print(f"  skip {fp.name}: {e}")
    return pd.DataFrame(records)


df_cp_text  = load_json_articles("papers_text")
df_nat_text = load_json_articles("papers_nature")
df_all      = pd.concat([df_cp_text, df_nat_text], ignore_index=True)

print(f"CellPress articles: {len(df_cp_text)}")
print(f"Nature articles:    {len(df_nat_text)}")
print(f"Combined:           {len(df_all)}")
df_all.head()

### 10.4 Export the combined DataFrame to CSV

In [ ]:
out_csv = "combined_articles.csv"
df_all.to_csv(out_csv, index=False)
print(f"Saved {len(df_all)} rows to {out_csv}.")

---
## 11 — Progress callbacks — custom hooks

All crawl functions accept two optional callbacks so you can integrate them with your own UI or logging pipeline.

### 11.1 `progress_callback(filename, filepath)`

Called **once per article** right after it is saved.

In [ ]:
saved_files = []

def on_article_saved(filename: str, filepath: str):
    """Custom hook: called with (filename, filepath) after each article is saved."""
    size_kb = os.path.getsize(filepath) / 1024 if os.path.exists(filepath) else 0
    print(f"  [SAVED] {filename}  ({size_kb:.1f} KB)")
    saved_files.append(filepath)


json_paths, titles = await crawl_text_async(
    year_from         = 2024,
    year_to           = 2024,
    out_folder        = "papers_text",
    headless          = True,
    limit             = 2,
    journal_slugs     = ["cell"],
    progress_callback = on_article_saved,   # <-- attach our hook
)

print(f"\nTotal saved via callback: {len(saved_files)}")

### 11.2 `total_progress_callback(current, total, status, file_size, speed, stage)`

Called **repeatedly** during crawling with overall progress information.

| Argument | Type | Description |
|----------|------|-------------|
| `current` | `int` | Number of articles processed so far |
| `total` | `int` | Total number of articles to process |
| `status` | `str` | Human-readable status message |
| `file_size` | `float` | Size of the last file in KB |
| `speed` | `float` | Current download/extraction speed in KB/s |
| `stage` | `str` | Current stage, e.g. `"downloading"`, `"extracting"` |

In [ ]:
from IPython.display import clear_output, display
import ipywidgets as widgets

progress_bar = widgets.IntProgress(
    min=0, max=100, value=0,
    description="Progress:",
    bar_style="info",
    style={"bar_color": "#2196F3"},
    layout=widgets.Layout(width="80%")
)
status_label = widgets.Label(value="Waiting to start...")

display(progress_bar, status_label)


def on_total_progress(current, total, status, file_size, speed, stage):
    """Update the ipywidgets progress bar."""
    if total and total > 0:
        progress_bar.max   = total
        progress_bar.value = current
    status_label.value = (
        f"{stage} | {status} | "
        f"{current}/{total} | "
        f"{file_size:.1f} KB | "
        f"{speed:.1f} KB/s"
    )


json_paths, titles = await crawl_text_async(
    year_from               = 2024,
    year_to                 = 2024,
    out_folder              = "papers_text",
    headless                = True,
    limit                   = 3,
    journal_slugs           = ["immunity"],
    total_progress_callback = on_total_progress,  # <-- attach our progress bar hook
)

progress_bar.bar_style = "success"
status_label.value = f"Done — {len(json_paths)} article(s) extracted."

### 11.3 Simple text-based progress (no widgets)

In [ ]:
def simple_progress(current, total, status, file_size, speed, stage):
    bar_len = 30
    filled  = int(bar_len * current / max(total, 1))
    bar     = "█" * filled + "░" * (bar_len - filled)
    print(f"\r[{bar}] {current}/{total}  {stage}: {status}", end="", flush=True)


json_paths, titles = await crawl_text_nature_async(
    year_from               = 2024,
    year_to                 = 2024,
    out_folder              = "papers_nature",
    headless                = True,
    limit                   = 3,
    journal_slugs           = ["nature-medicine"],
    total_progress_callback = simple_progress,
)
print()  # newline after inline progress
print(f"Done — {len(json_paths)} article(s).")

---
## 12 — Troubleshooting

### 12.1 Browser not installed

```
playwright._impl._errors.Error: Executable doesn't exist at ...
```

**Fix:** Re-run Section 1.2 to install the Firefox browser binaries.

---

### 12.2 Cloudflare challenge / CAPTCHA detected

The crawler logs a warning when a Cloudflare bot-protection page is detected. In that case:

1. Set `headless=False` to open a visible browser window.
2. Manually complete the CAPTCHA once.
3. The crawl will resume automatically.

```python
await crawl_text_async(
    headless      = False,   # <-- visible browser
    limit         = 3,
    journal_slugs = ["cell"],
    year_from     = 2024,
    year_to       = 2024,
)
```

---

### 12.3 `RuntimeError: This event loop is already running`

This means you tried to call `asyncio.run()` inside Jupyter.  
**Fix:** Use `await` directly instead:

```python
# WRONG in Jupyter:
asyncio.run(crawl_text_async(...))

# CORRECT in Jupyter:
result = await crawl_text_async(...)
```

---

### 12.4 Empty results / no articles found

- Double-check the journal slug (use `discover_journals_async()` to list valid slugs).
- Widen the year range — some journals may not have published in the specified window.
- Try `force_refresh=True` in the journal discovery call to update the cache.

---

### 12.5 Force-refresh the journal cache

In [ ]:
# Force a fresh fetch from Cell.com / Nature.com to update the journal list
fresh_cp  = await discover_journals_async(force_refresh=True)
fresh_nat = await discover_journals_nature_async(force_refresh=True)

print(f"CellPress: {len(fresh_cp)} journals")
print(f"Nature:    {len(fresh_nat)} journals")

---

## Summary

| Task | Function | Key params |
|------|----------|------------|
| Discover CellPress journals | `discover_journals()` / `await discover_journals_async()` | `force_refresh` |
| Discover Nature journals | `await discover_journals_nature_async()` | `force_refresh` |
| Download CellPress PDFs (script) | `crawl(...)` | `keywords`, `year_from/to`, `journal_slugs`, `limit`, `out_folder` |
| Download CellPress PDFs (Jupyter) | `await crawl_async(...)` | same + `crawl_archives` |
| Extract CellPress full-text JSON | `await crawl_text_async(...)` | same as above |
| Extract Nature full-text JSON | `await crawl_text_nature_async(...)` | same (no keyword support yet) |

Happy crawling! 🎉